# Date Dimension Loader

Generates and maintains the `warehouse.dim_date` dimension table with incremental refresh capability.

## Purpose
Comprehensive date dimension covering historical and future dates for time-based analysis.

## Key Features
* Date keys in YYYYMMDD format for efficient joins
* Calendar attributes (year, quarter, month, week)
* Business day flags and fiscal period support
* Holiday calendar integration ready
* Incremental updates via MERGE (only new date ranges)

## Architecture
**Target**: `workspace.warehouse.dim_date`  
**Metadata**: `workspace.metadata.dim_date_refresh_log`  
**Mode**: Incremental (merge new dates, idempotent)

## Batch Processing
* Tracks refresh history in metadata table
* Only inserts new date records (merge based on date_sk)
* Safe to re-run with same or extended date ranges
* Validates data quality after each refresh

In [0]:
dbutils.widgets.text("start_date", "2020-01-01", "Start Date (YYYY-MM-DD)")
dbutils.widgets.text("end_date", "2030-12-31", "End Date (YYYY-MM-DD)")
dbutils.widgets.dropdown("fiscal_year_start_month", "7", ["1", "4", "7", "10"], "Fiscal Year Start Month")
dbutils.widgets.dropdown("force_full_refresh", "false", ["true", "false"], "Force Full Refresh")

START_DATE = dbutils.widgets.get("start_date")
END_DATE = dbutils.widgets.get("end_date")
FISCAL_YEAR_START_MONTH = int(dbutils.widgets.get("fiscal_year_start_month"))
FORCE_FULL_REFRESH = dbutils.widgets.get("force_full_refresh") == "true"

In [0]:
from datetime import datetime, timedelta
from pyspark.sql import Row, functions as F
from pyspark.sql.types import *
import pandas as pd
import json

CATALOG = "workspace"
WAREHOUSE_SCHEMA = f"{CATALOG}.warehouse"
METADATA_SCHEMA = f"{CATALOG}.metadata"

TARGET_TABLE = f"{WAREHOUSE_SCHEMA}.dim_date"
METADATA_TABLE = f"{METADATA_SCHEMA}.dim_date_refresh_log"

run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
run_timestamp = datetime.now()

print(f"Run ID: {run_id}")
print(f"Date range: {START_DATE} to {END_DATE}")
print(f"Fiscal year starts: Month {FISCAL_YEAR_START_MONTH}")
print(f"Force full refresh: {FORCE_FULL_REFRESH}")

In [0]:
%sql
-- Create date dimension table if not exists
CREATE TABLE IF NOT EXISTS workspace.warehouse.dim_date (
  date_sk INT NOT NULL COMMENT 'Date key YYYYMMDD',
  date_value DATE NOT NULL COMMENT 'Actual date',
  year INT NOT NULL COMMENT 'Year',
  quarter INT NOT NULL COMMENT 'Quarter 1-4',
  month INT NOT NULL COMMENT 'Month 1-12',
  month_name STRING NOT NULL COMMENT 'Month name',
  day_of_month INT NOT NULL COMMENT 'Day of month',
  day_of_week INT NOT NULL COMMENT 'Day of week 1-7',
  day_of_week_name STRING NOT NULL COMMENT 'Day name',
  day_of_year INT NOT NULL COMMENT 'Day of year',
  week_of_year INT NOT NULL COMMENT 'ISO week number',
  is_weekend BOOLEAN NOT NULL COMMENT 'Is weekend day',
  is_month_start BOOLEAN NOT NULL COMMENT 'First day of month',
  is_month_end BOOLEAN NOT NULL COMMENT 'Last day of month',
  is_quarter_start BOOLEAN NOT NULL COMMENT 'First day of quarter',
  is_quarter_end BOOLEAN NOT NULL COMMENT 'Last day of quarter',
  is_year_start BOOLEAN NOT NULL COMMENT 'First day of year',
  is_year_end BOOLEAN NOT NULL COMMENT 'Last day of year',
  fiscal_year INT NOT NULL COMMENT 'Fiscal year',
  fiscal_quarter INT NOT NULL COMMENT 'Fiscal quarter',
  CONSTRAINT pk_dim_date PRIMARY KEY (date_sk)
)
USING DELTA
COMMENT 'Date dimension for time-based analysis';

-- Create metadata tracking table
CREATE TABLE IF NOT EXISTS workspace.metadata.dim_date_refresh_log (
  run_id STRING,
  start_date DATE,
  end_date DATE,
  fiscal_year_start_month INT,
  dates_generated INT,
  dates_inserted INT,
  dates_updated INT,
  force_full_refresh BOOLEAN,
  processed_at TIMESTAMP,
  status STRING,
  error_message STRING
)
USING DELTA
COMMENT 'Tracks date dimension refresh history';

In [0]:
# Calculate fiscal year and quarter based on fiscal year start month
def calculate_fiscal_year(month, year):
    if month >= FISCAL_YEAR_START_MONTH:
        return year
    else:
        return year - 1

def calculate_fiscal_quarter(month):
    # Calculate quarter based on fiscal year start
    adjusted_month = (month - FISCAL_YEAR_START_MONTH) % 12
    return (adjusted_month // 3) + 1

# Generate date range
start = datetime.strptime(START_DATE, '%Y-%m-%d')
end = datetime.strptime(END_DATE, '%Y-%m-%d')
date_range = pd.date_range(start, end, freq='D')

print(f"Generating {len(date_range):,} date records...")

# Create date dimension with all attributes
date_data = []
for date in date_range:
    date_data.append(Row(
        date_sk=int(date.strftime('%Y%m%d')),
        date_value=date.date(),
        year=date.year,
        quarter=date.quarter,
        month=date.month,
        month_name=date.strftime('%B'),
        day_of_month=date.day,
        day_of_week=date.dayofweek + 1,  # 1=Monday, 7=Sunday
        day_of_week_name=date.strftime('%A'),
        day_of_year=date.dayofyear,
        week_of_year=date.isocalendar()[1],
        is_weekend=(date.dayofweek >= 5),
        is_month_start=(date.day == 1),
        is_month_end=(date.day == pd.Period(date, freq='D').days_in_month),
        is_quarter_start=(date.month % 3 == 1 and date.day == 1),
        is_quarter_end=(date.month % 3 == 0 and date.day == pd.Period(date, freq='D').days_in_month),
        is_year_start=(date.month == 1 and date.day == 1),
        is_year_end=(date.month == 12 and date.day == 31),
        fiscal_year=calculate_fiscal_year(date.month, date.year),
        fiscal_quarter=calculate_fiscal_quarter(date.month)
    ))

# Create DataFrame
date_df = spark.createDataFrame(date_data)

print(f"✓ Generated {date_df.count():,} date records")
print(f"  Date range: {date_df.agg(F.min('date_value')).collect()[0][0]} to {date_df.agg(F.max('date_value')).collect()[0][0]}")

# Create temp view for merge
date_df.createOrReplaceTempView('date_temp')

In [0]:
# Define metadata schema explicitly to match table definition
from pyspark.sql.types import StructType, StructField, StringType, DateType, IntegerType, BooleanType, TimestampType

metadata_schema = StructType([
    StructField("run_id", StringType(), True),
    StructField("start_date", DateType(), True),
    StructField("end_date", DateType(), True),
    StructField("fiscal_year_start_month", IntegerType(), True),
    StructField("dates_generated", IntegerType(), True),
    StructField("dates_inserted", IntegerType(), True),
    StructField("dates_updated", IntegerType(), True),
    StructField("force_full_refresh", BooleanType(), True),
    StructField("processed_at", TimestampType(), True),
    StructField("status", StringType(), True),
    StructField("error_message", StringType(), True)
])

try:
    print(f"Merging date records into {TARGET_TABLE}...", end=" ")
    
    if FORCE_FULL_REFRESH:
        # Full refresh: truncate and insert all
        spark.sql(f"TRUNCATE TABLE {TARGET_TABLE}")
        date_df.write.format("delta").mode("append").saveAsTable(TARGET_TABLE)
        dates_inserted = date_df.count()
        dates_updated = 0
        print(f"✓ Full refresh: {dates_inserted:,} records inserted")
    else:
        # Incremental: merge only new dates
        merge_sql = f"""
        MERGE INTO {TARGET_TABLE} target
        USING date_temp source
        ON target.date_sk = source.date_sk
        WHEN NOT MATCHED THEN INSERT *
        """
        
        spark.sql(merge_sql)
        
        # Count metrics
        dates_inserted = spark.sql(f"""
            SELECT COUNT(*) as cnt FROM date_temp 
            WHERE date_sk NOT IN (SELECT date_sk FROM {TARGET_TABLE})
        """).collect()[0]['cnt']
        dates_updated = 0
        
        print(f"✓ Merge complete: {dates_inserted:,} new records inserted")
    
    # Log to metadata with explicit schema
    metadata_data = [(
        run_id,
        datetime.strptime(START_DATE, '%Y-%m-%d').date(),
        datetime.strptime(END_DATE, '%Y-%m-%d').date(),
        FISCAL_YEAR_START_MONTH,
        date_df.count(),
        dates_inserted,
        dates_updated,
        FORCE_FULL_REFRESH,
        run_timestamp,
        'success',
        None
    )]
    
    metadata_record = spark.createDataFrame(metadata_data, schema=metadata_schema)
    metadata_record.write.format("delta").mode("append").saveAsTable(METADATA_TABLE)
    
    result = {
        "status": "success",
        "run_id": run_id,
        "dates_generated": date_df.count(),
        "dates_inserted": dates_inserted,
        "dates_updated": dates_updated,
        "target_table": TARGET_TABLE,
        "metadata_table": METADATA_TABLE
    }
    
    print(json.dumps(result, indent=2))
    
except Exception as e:
    error_msg = str(e)
    print(f"✗ Error: {error_msg}")
    
    # Log failure to metadata with explicit schema
    metadata_data = [(
        run_id,
        datetime.strptime(START_DATE, '%Y-%m-%d').date(),
        datetime.strptime(END_DATE, '%Y-%m-%d').date(),
        FISCAL_YEAR_START_MONTH,
        date_df.count() if 'date_df' in locals() else 0,
        0,
        0,
        FORCE_FULL_REFRESH,
        run_timestamp,
        'failed',
        error_msg
    )]
    
    metadata_record = spark.createDataFrame(metadata_data, schema=metadata_schema)
    metadata_record.write.format("delta").mode("append").saveAsTable(METADATA_TABLE)
    
    raise

In [0]:
%sql
-- Validate date dimension
SELECT 
  COUNT(*) as total_dates,
  MIN(date_value) as earliest_date,
  MAX(date_value) as latest_date,
  COUNT(DISTINCT year) as years,
  COUNT(DISTINCT fiscal_year) as fiscal_years,
  SUM(CASE WHEN is_weekend THEN 1 ELSE 0 END) as weekend_days,
  SUM(CASE WHEN NOT is_weekend THEN 1 ELSE 0 END) as weekdays,
  SUM(CASE WHEN is_month_start THEN 1 ELSE 0 END) as month_starts,
  SUM(CASE WHEN is_quarter_start THEN 1 ELSE 0 END) as quarter_starts,
  SUM(CASE WHEN is_year_start THEN 1 ELSE 0 END) as year_starts
FROM workspace.warehouse.dim_date;

-- Show refresh history
SELECT 
  run_id,
  start_date,
  end_date,
  fiscal_year_start_month,
  dates_generated,
  dates_inserted,
  force_full_refresh,
  processed_at,
  status
FROM workspace.metadata.dim_date_refresh_log
ORDER BY processed_at DESC
LIMIT 10;